# L5c Example: Apples, Oranges, and Linear Allocation

> **Learning objectives**
>
> - Identify decision variables, objective coefficients, and a budget constraint.
> - Predict an LP solution from utility per dollar.
> - Recognize alternate optima when objective and constraint slopes align.
> - Validate solver output using expenditure and objective calculations.


## Setup

Run the local setup cell first. It activates the pinned course environment, loads every package used by this meeting, and includes the `L5cLinearPrograms` module from [`src/LinearPrograms.jl`](src/LinearPrograms.jl), which provides the functions called below.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and includes the meeting's local source. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


## Model

For quantities $x_A,x_O\ge0$, utilities $u_A,u_O$, prices $p_A,p_O$, and budget $I$:

$$\max\;u_Ax_A+u_Ox_O\quad\text{s.t.}\quad p_Ax_A+p_Ox_O\le I.$$

The ratio $u_i/p_i$ is utility per dollar. With one resource constraint, the larger ratio predicts the preferred corner; equal ratios create a full edge of optimal solutions.


In [2]:
prices = [2.0, 4.0]
budget = 100.0
cases = [
    (name = "A: apples dominate", utilities = [0.55, 0.45]),
    (name = "B: oranges dominate", utilities = [0.15, 0.55]),
    (name = "C: equal ratios", utilities = [2.0, 4.0]),
]


3-element Vector{@NamedTuple{name::String, utilities::Vector{Float64}}}:
 (name = "A: apples dominate", utilities = [0.55, 0.45])
 (name = "B: oranges dominate", utilities = [0.15, 0.55])
 (name = "C: equal ratios", utilities = [2.0, 4.0])

## Solve and compare three geometric cases


In [3]:
solutions = [solve_fruit_problem(case.utilities, prices, budget) for case in cases]
comparison = DataFrame(
    case = [case.name for case in cases],
    utility_per_dollar_apples = [solution.utility_per_dollar[1] for solution in solutions],
    utility_per_dollar_oranges = [solution.utility_per_dollar[2] for solution in solutions],
    apples = [solution.quantities[1] for solution in solutions],
    oranges = [solution.quantities[2] for solution in solutions],
    expenditure = [solution.expenditure for solution in solutions],
    objective = [solution.utility for solution in solutions],
)
pretty_table(comparison)


┌─────────────────────┬───────────────────────────┬─────────────────────────────
│                case │ utility_per_dollar_apples │ utility_per_dollar_oranges ⋯
│              String │                   Float64 │                    Float64 ⋯
├─────────────────────┼───────────────────────────┼─────────────────────────────
│  A: apples dominate │                     0.275 │                     0.1125 ⋯
│ B: oranges dominate │                     0.075 │                     0.1375 ⋯
│     C: equal ratios │                       1.0 │                        1.0 ⋯
└─────────────────────┴───────────────────────────┴─────────────────────────────
                                                               4 columns omitted


![Three objective/constraint slope cases](figs/Fig-ThreeCases-LP-Schematic.svg)


## Verification contracts


In [4]:
@test solutions[1].quantities ≈ [50.0, 0.0]
@test solutions[2].quantities ≈ [0.0, 25.0]
@test all(solution.expenditure <= budget + 1e-8 for solution in solutions)
@test solutions[3].utility ≈ 100.0
@test dot(prices, solutions[3].quantities) ≈ budget
:allocation_cases_verified


:allocation_cases_verified

## Alternate optima are not solver disagreement

In Case C, $u_A/p_A=u_O/p_O=1$. Every nonnegative point satisfying $2x_A+4x_O=100$ has objective value 100. A solver may return either corner or another point on that edge; validate the objective and constraint rather than demanding one exact quantity vector.


## Summary

Utility per dollar predicts the corner solution for this one-resource model. Equal ratios create alternate optima, illustrating why model interpretation must distinguish a unique decision from a unique objective value.
